# Local alternative feature summary

This notebook demonstrates `plotly.local.alternative_feature_summary` for one local alternative explanation. The plot is a local summary: it shows how often each feature participates in the alternatives for one explained instance. It is not global feature importance.

The main stacked bars show primary role plus quality-flag combinations such as `counter + ensured`, `counter + pareto`, and `counter + ensured + pareto`. `ensured` and `pareto` are quality flags represented inside the role bars, not a separate default status panel.

The optional conjunction panel is disabled by default. When enabled, it counts how often a feature participates in multi-feature rules. Unknown roles mean the role metadata was unavailable or unmapped, not that the rule has no role.

In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import subprocess
import sys
from pathlib import Path

package_dir = Path.cwd().resolve()
if package_dir.name == 'examples':
    package_dir = package_dir.parent
else:
    repo_candidate = Path('packages/visualization/calibrated-explanations-visualization-plotly').resolve()
    if repo_candidate.exists():
        package_dir = repo_candidate

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])

0

In [17]:
import importlib
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from calibrated_explanations import WrapCalibratedExplainer
import calibrated_explanations.plugins.registry as registry
import ce_visualization_plotly.alternative_feature_summary as alternative_feature_summary_module
import ce_visualization_plotly.plugin as plotly_plugin_module

reset_catalog = getattr(registry, 'reset_plugin_catalog', None)
if callable(reset_catalog):
    reset_catalog(kind='all')
clear_env_cache = getattr(registry, 'clear_env_trust_cache', None)
if callable(clear_env_cache):
    clear_env_cache()
clear_warnings = getattr(registry, 'clear_trust_warnings', None)
if callable(clear_warnings):
    clear_warnings()
for module_name in [name for name in list(sys.modules) if name.startswith('ce_visualization_plotly')]:
    sys.modules.pop(module_name, None)
import ce_visualization_plotly.alternative_feature_summary as alternative_feature_summary_module
import ce_visualization_plotly.plugin as plotly_plugin_module
importlib.reload(alternative_feature_summary_module)
importlib.reload(plotly_plugin_module)
plotly_plugin_module.register_plotly_visualization_components()
np.set_printoptions(precision=3, suppress=True)

In [18]:
X, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_redundant=0,
    random_state=0,
)

x_proper, x_holdout, y_proper, y_holdout = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=0,
    stratify=y,
)
x_cal, X_query, y_cal, y_query = train_test_split(
    x_holdout,
    y_holdout,
    test_size=0.5,
    random_state=0,
    stratify=y_holdout,
)

assert len(x_proper) == 300
assert len(x_cal) == 100
assert len(X_query) == 100

In [19]:
model = RandomForestClassifier(n_estimators=100, random_state=0)
explainer = WrapCalibratedExplainer(model)
explainer.fit(x_proper, y_proper)
assert explainer.fitted is True

explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

In [20]:
alternatives = explainer.explore_alternatives(X_query[:5])

try:
    alternatives.add_conjunctions(max_rule_size=3)
except AttributeError:
    try:
        alternatives[0].add_conjunctions(max_rule_size=3)
    except AttributeError:
        pass

Default view: role-quality combinations only.

In [26]:
alt = alternatives[0].plot(style="plotly.local.alternative_feature_summary", show=True)

Limit the display to the most involved features.

In [27]:
alt = alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=True,
    filter_top_features=10,
)

Normalize each feature row to shares while preserving raw counts in hover.

In [ ]:
alt = alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=True,
    normalize="share",
)

PlotRenderResult(artifact={'artifact_type': 'plotly.local.alternative_feature_summary', 'artifact_version': '0.2.0', 'style': 'plotly.local.alternative_feature_summary', 'mode': 'classification', 'task': 'classification', 'instance_metadata': {'instance_index': 0, 'prediction': {'predict': 0.7499999999999999, 'low': 0.6666666666666666, 'high': 1.0, 'classes': 1.0, '__full_probabilities__': array([[0.25 , 0.75 ],
       [0.033, 0.967],
       [0.05 , 0.95 ],
       [0.071, 0.929],
       [0.848, 0.152]])}}, 'rule_records': [{'rule_id': 'rule-0 (rank 1)', 'rank': 1, 'feature_indices': [0], 'feature_names': ['0'], 'true_values': [-0.4224987168254349], 'rule': '0 > 0.40', 'rule_size': 1, 'is_conjunction': False, 'primary_role': 'semi', 'quality_flags': [], 'role_quality_key': 'semi', 'role_quality_label': 'semi', 'role_source': 'ce_metadata', 'is_counter': False, 'is_super': False, 'is_semi': True, 'is_ensured': False, 'is_pareto': False, 'prediction': 0.7222222222222222, 'low': 0.61111111

Enable the optional conjunction panel. Conjunction bars count how often a feature appears in multi-feature rules.

In [24]:
alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=True,
    include_conjunctions=True,
)

PlotRenderResult(artifact={'artifact_type': 'plotly.local.alternative_feature_summary', 'artifact_version': '0.2.0', 'style': 'plotly.local.alternative_feature_summary', 'mode': 'classification', 'task': 'classification', 'instance_metadata': {'instance_index': 0, 'prediction': {'predict': 0.7499999999999999, 'low': 0.6666666666666666, 'high': 1.0, 'classes': 1.0, '__full_probabilities__': array([[0.25 , 0.75 ],
       [0.033, 0.967],
       [0.05 , 0.95 ],
       [0.071, 0.929],
       [0.848, 0.152]])}}, 'rule_records': [{'rule_id': 'rule-0 (rank 1)', 'rank': 1, 'feature_indices': [0], 'feature_names': ['0'], 'true_values': [-0.4224987168254349], 'rule': '0 > 0.40', 'rule_size': 1, 'is_conjunction': False, 'primary_role': 'semi', 'quality_flags': [], 'role_quality_key': 'semi', 'role_quality_label': 'semi', 'role_source': 'ce_metadata', 'is_counter': False, 'is_super': False, 'is_semi': True, 'is_ensured': False, 'is_pareto': False, 'prediction': 0.7222222222222222, 'low': 0.61111111

Export to HTML without displaying the figure.

In [25]:
alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=False,
    path="alternative_feature_summary.html",
)

PlotRenderResult(artifact={'artifact_type': 'plotly.local.alternative_feature_summary', 'artifact_version': '0.2.0', 'style': 'plotly.local.alternative_feature_summary', 'mode': 'classification', 'task': 'classification', 'instance_metadata': {'instance_index': 0, 'prediction': {'predict': 0.7499999999999999, 'low': 0.6666666666666666, 'high': 1.0, 'classes': 1.0, '__full_probabilities__': array([[0.25 , 0.75 ],
       [0.033, 0.967],
       [0.05 , 0.95 ],
       [0.071, 0.929],
       [0.848, 0.152]])}}, 'rule_records': [{'rule_id': 'rule-0 (rank 1)', 'rank': 1, 'feature_indices': [0], 'feature_names': ['0'], 'true_values': [-0.4224987168254349], 'rule': '0 > 0.40', 'rule_size': 1, 'is_conjunction': False, 'primary_role': 'semi', 'quality_flags': [], 'role_quality_key': 'semi', 'role_quality_label': 'semi', 'role_source': 'ce_metadata', 'is_counter': False, 'is_super': False, 'is_semi': True, 'is_ensured': False, 'is_pareto': False, 'prediction': 0.7222222222222222, 'low': 0.61111111